# LLM Reasoning Analysis

## Configuration & Constants

In [ ]:
import pandas as pd
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string
from scipy import stats

# --- Global Settings for Easy Modification (Aligned with main_analysis.ipynb) ---
FIG_SIZE_SINGLE = (10, 7)     
FIG_SIZE_MULTI = (18, 14)     # Increased size for better padding
FONT_SCALE = 2.0
DPI = 300

# Define consistent colors for models
MODEL_NAME_MAPPING = {
    'x-ai/grok-4-fast': 'Grok',
    'openai/gpt-5-nano': 'ChatGPT',
    'google/gemini-2.5-flash': 'Gemini',
    'deepseek/deepseek-chat-v3.1': 'DeepSeek'
}

# Create a consistent color palette
unique_models = sorted(list(MODEL_NAME_MAPPING.values()))
_palette_raw = sns.color_palette("tab10", n_colors=len(unique_models))
MODEL_COLORS = dict(zip(unique_models, _palette_raw))

# Publication-ready style
sns.set_style("white")
sns.set_context("paper", font_scale=FONT_SCALE)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['figure.dpi'] = DPI

## Data Loading

In [ ]:
strategies = {
    'strategy_one': 'strategy_one_results',
    'strategy_two': 'strategy_two_results',
    'strategy_three': 'strategy_three_results',
    'strategy_four': 'strategy_four_results'
}

all_dfs = []

for strategy_name, directory in strategies.items():
    dir_path = os.path.join('llm_judge_results', directory)
    if not os.path.exists(dir_path):
        print(f"Warning: Directory '{dir_path}' not found. Skipping...")
        continue
        
    csv_files = glob.glob(os.path.join(dir_path, '*judged.csv'))
    
    for csv_file in csv_files:
        df_temp = pd.read_csv(csv_file)
        df_temp['strategy'] = strategy_name
        all_dfs.append(df_temp)

if all_dfs:
    df = pd.concat(all_dfs, ignore_index=True)
    print(f"Successfully combined {len(all_dfs)} files. Total rows: {len(df)}")
else:
    print("No data files were found or loaded.")
    df = pd.DataFrame() 

if not df.empty:
    df['model'] = df['model'].replace(MODEL_NAME_MAPPING)
    
    # Calculate irrational choice using the same logic as main_analysis.ipynb
    df['irrational_choice'] = (
        (df['infection_rate_selected'] > df['infection_rate_non_selected']) &
        (df['npi_level_selected'] >= df['npi_level_non_selected'])
    )
    
    df['choice_status'] = np.where(df['irrational_choice'], "Irrational", "Rational")

    # Define reasoning outcome variables (columns of interest)
    columns_of_interest = [
        'mentions_hospital_capacity',
        'computed_hospital_demand',
        'compares_demand_to_capacity', 
        'calculated_hospital_shortfall',
        'economic_impact_depth', 
        'mentions_business_closures', 
        'compares_infection_rates_between_options', 
        'argues_closures_not_worth_it_given_infections', 
        'mentions_user_preference'
    ]
    
    # Calculate reasoning length
    df['reasoning_length'] = df['reasoning'].str.len().fillna(0)
    
    # Ensure all columns of interest are present and numeric
    for col in columns_of_interest:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        else:
            print(f"Warning: Column '{col}' not found in data.")

## Helper Functions

In [ ]:
def add_panel_labels(axes, labels=None, x_offset=-0.02, y_offset=1.05):
    """
    Add letter labels (A, B, C...) and optional subtitles to subplots.
    Aligned with main_analysis.ipynb style but preserved subtitle functionality.
    """
    axes_flat = axes.flatten() if isinstance(axes, np.ndarray) else [axes]
    for i, ax in enumerate(axes_flat):
        if ax.get_visible():
            char = string.ascii_uppercase[i]
            text = char
            if labels and i < len(labels):
                text = f"{char}) {labels[i]}"
            
            ax.text(x_offset, y_offset, text, transform=ax.transAxes,
                    fontsize=26, fontweight='bold', va='bottom', ha='right' if not labels else 'left')

def get_strategy_label(strat):
    return strat.replace('_', ' ').title()

def get_column_label(col):
    mapping = {
        'mentions_hospital_capacity': 'Mentions Hospital Capacity',
        'computed_hospital_demand': 'Computes Hospital Demand',
        'compares_demand_to_capacity': 'Compares Demand to Capacity',
        'calculated_hospital_shortfall': 'Calculates Shortfall',
        'economic_impact_depth': 'Mean Economic Depth',
        'mentions_business_closures': 'Mentions Closures',
        'compares_infection_rates_between_options': 'Compares Infections',
        'argues_closures_not_worth_it_given_infections': 'Argues Cost/Benefit',
        'mentions_user_preference': 'Mentions User Preference'
    }
    return mapping.get(col, col.replace('_', ' ').title())

## Analysis & Figures

### Figure 1: Reasoning Components Frequency by Model
This figure shows the frequency with which different models included specific reasoning components in their decision-making process.

In [ ]:
# Reshape the data for plotting
df_reasoning = df.melt(
    id_vars=['model', 'strategy', 'choice_status'], 
    value_vars=columns_of_interest, 
    var_name='component', 
    value_name='present'
)
df_reasoning['component_label'] = df_reasoning['component'].apply(get_column_label)

plt.figure(figsize=FIG_SIZE_SINGLE)
ax = sns.barplot(
    data=df_reasoning,
    x='present', 
    y='component_label', 
    hue='model', 
    palette=MODEL_COLORS,
    errorbar=None
)

plt.xlabel('Frequency (Mean)')
plt.ylabel('')
plt.title('Figure 1: Inclusion Frequency of Reasoning Components by Model', fontsize=26, pad=20, fontweight='bold')
plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=14, title_fontsize=16)
ax.tick_params(axis='y', labelsize=14)
sns.despine()
plt.tight_layout()
plt.show()

### Figure 2: Reasoning Patterns in Rational vs. Irrational Choices across Models
We examine whether models that make irrational choices show different reasoning patterns compared to their rational decisions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=FIG_SIZE_MULTI, sharex=True)
axes = axes.flatten()

model_order = sorted(df['model'].unique())

for i, model_name in enumerate(model_order):
    ax = axes[i]
    model_data = df_reasoning[df_reasoning['model'] == model_name]
    
    sns.barplot(
        data=model_data,
        x='present', 
        y='component_label', 
        hue='choice_status', 
        ax=ax,
        palette={'Rational': 'gray', 'Irrational': 'red'},
        errorbar=None
    )
    
    ax.set_xlabel('Frequency' if i >= 2 else '')
    ax.set_ylabel('')
    ax.tick_params(axis='y', labelsize=14)
    if ax.get_legend():
        ax.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Choice Status', loc='center right', bbox_to_anchor=(1.15, 0.5), fontsize=14, title_fontsize=16)

fig.suptitle('Figure 2: Reasoning Patterns in Rational vs. Irrational Choices across Models', fontsize=28, y=1.02, fontweight='bold')
add_panel_labels(axes, labels=model_order)
sns.despine()
plt.tight_layout()
plt.show()

### Figure 3: Key Reasoning Components across Prompting Strategies
Different prompting strategies may prioritize different reasoning elements.

In [ ]:
# Focusing on a subset of the most relevant components to maintain readability
subset_cols = [
    'mentions_hospital_capacity', 
    'compares_demand_to_capacity', 
    'economic_impact_depth', 
    'compares_infection_rates_between_options'
]

fig, axes = plt.subplots(2, 2, figsize=FIG_SIZE_MULTI, sharex=True)
axes = axes.flatten()

label_list = [get_column_label(col) for col in subset_cols]

for i, col in enumerate(subset_cols):
    ax = axes[i]
    sns.barplot(
        data=df,
        x='strategy', 
        y=col, 
        hue='model', 
        ax=ax, 
        palette=MODEL_COLORS,
        order=['strategy_one', 'strategy_two', 'strategy_three', 'strategy_four'],
        errorbar=None
    )
    
    ax.set_xticklabels([get_strategy_label(s) for s in ['strategy_one', 'strategy_two', 'strategy_three', 'strategy_four']], rotation=45)
    ax.set_xlabel('')
    ax.set_ylabel('Frequency')
    if ax.get_legend():
        ax.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Model', loc='center right', bbox_to_anchor=(1.12, 0.5), fontsize=14, title_fontsize=16)

fig.suptitle('Figure 3: Key Reasoning Components across Prompting Strategies', fontsize=28, y=1.02, fontweight='bold')
add_panel_labels(axes, labels=label_list)
sns.despine()
plt.tight_layout()
plt.show()

### Figure 4: Average Reasoning Length by Model and Strategy
We analyze the verbosity of model outputs across different prompting conditions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=FIG_SIZE_MULTI, sharey=True)
axes = axes.flatten()

strat_order = ['strategy_one', 'strategy_two', 'strategy_three', 'strategy_four']
strat_labels = [get_strategy_label(s) for s in strat_order]

for i, strat in enumerate(strat_order):
    ax = axes[i]
    strat_data = df[df['strategy'] == strat]
    
    sns.barplot(
        data=strat_data,
        x='model', 
        y='reasoning_length', 
        ax=ax, 
        palette=MODEL_COLORS,
        errorbar=None
    )
    
    ax.set_xlabel('')
    ax.set_ylabel('Characters')
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Figure 4: Average Reasoning Length by Model and Strategy', fontsize=28, y=1.02, fontweight='bold')
add_panel_labels(axes, labels=strat_labels)
sns.despine()
plt.tight_layout()
plt.show()